In [7]:
import pandas as pd

from sklearn.preprocessing import StandardScaler

In [5]:
df_15min = pd.read_csv('../data/afeka/germany_thin.csv', parse_dates=['utc_timestamp'])
df_15min

,utc_timestamp,load_actual_entsoe_transparency,load_forecast_entsoe_transparency
0,2015-01-01 00:15:00+00:00,41517.72,40204.18
1,2015-01-01 00:30:00+00:00,41179.17,39640.26
2,2015-01-01 00:45:00+00:00,40756.40,39323.88
3,2015-01-01 01:00:00+00:00,40617.76,39002.42
4,2015-01-01 01:15:00+00:00,40312.25,38869.21
...,...,...,...
201295,2020-09-30 22:30:00+00:00,46505.50,46360.69
201296,2020-09-30 22:45:00+00:00,46229.48,45971.68
201297,2020-09-30 23:00:00+00:00,45792.82,45303.45
201298,2020-09-30 23:15:00+00:00,45471.18,44987.77


In [24]:
# copy dataset without column 'load_forecast_entsoe_transparency'
data = df_15min.drop(columns=['load_forecast_entsoe_transparency'])

# Extract time-related features
data['hour'] = data['utc_timestamp'].dt.hour
data['dayofweek'] = data['utc_timestamp'].dt.dayofweek
data['month'] = data['utc_timestamp'].dt.month

In [25]:
from workalendar.europe import BadenWurttemberg

# Initialize the calendar
calendar = BadenWurttemberg()

# Check if each date is a workday
data['is_workday'] = data['utc_timestamp'].apply(lambda dt: int(calendar.is_working_day(dt)))
data['is_holiday'] = data['utc_timestamp'].apply(lambda dt: int(calendar.is_holiday(dt)))

data

,utc_timestamp,load_actual_entsoe_transparency,hour,dayofweek,month,is_workday,is_holiday
0,2015-01-01 00:15:00+00:00,41517.72,0,3,1,0,1
1,2015-01-01 00:30:00+00:00,41179.17,0,3,1,0,1
2,2015-01-01 00:45:00+00:00,40756.40,0,3,1,0,1
3,2015-01-01 01:00:00+00:00,40617.76,1,3,1,0,1
4,2015-01-01 01:15:00+00:00,40312.25,1,3,1,0,1
...,...,...,...,...,...,...,...
201295,2020-09-30 22:30:00+00:00,46505.50,22,2,9,1,0
201296,2020-09-30 22:45:00+00:00,46229.48,22,2,9,1,0
201297,2020-09-30 23:00:00+00:00,45792.82,23,2,9,1,0
201298,2020-09-30 23:15:00+00:00,45471.18,23,2,9,1,0


In [26]:
# Normalize load values
scaler = StandardScaler()
data[['load_actual_entsoe_transparency']] = scaler.fit_transform(data[['load_actual_entsoe_transparency']])
data

,utc_timestamp,load_actual_entsoe_transparency,hour,dayofweek,month,is_workday,is_holiday
0,2015-01-01 00:15:00+00:00,-1.391060,0,3,1,0,1
1,2015-01-01 00:30:00+00:00,-1.424778,0,3,1,0,1
2,2015-01-01 00:45:00+00:00,-1.466884,0,3,1,0,1
3,2015-01-01 01:00:00+00:00,-1.480691,1,3,1,0,1
4,2015-01-01 01:15:00+00:00,-1.511118,1,3,1,0,1
...,...,...,...,...,...,...,...
201295,2020-09-30 22:30:00+00:00,-0.894305,22,2,9,1,0
201296,2020-09-30 22:45:00+00:00,-0.921795,22,2,9,1,0
201297,2020-09-30 23:00:00+00:00,-0.965284,23,2,9,1,0
201298,2020-09-30 23:15:00+00:00,-0.997317,23,2,9,1,0


In [27]:
# Extract values from dataset
X_values = data[['load_actual_entsoe_transparency']].values
hour_values = data['hour'].values
day_values = data['dayofweek'].values
month_values = data['month'].values
holiday_values = data['is_holiday'].values
workday_values = data['is_workday'].values

# Prepare sequences for training
seq_length = 20
X, hour_list, day_list, month_list, holiday_list, workday_list, y = [], [], [], [], [], [], []

for i in range(len(data) - seq_length):
    X.append(X_values[i:i+seq_length])
    hour_list.append(hour_values[i:i+seq_length])
    day_list.append(day_values[i:i+seq_length])
    month_list.append(month_values[i:i+seq_length])
    holiday_list.append(holiday_values[i:i+seq_length])
    workday_list.append(workday_values[i:i+seq_length])
    y.append(X_values[i+seq_length])  # Target load


In [37]:
import torch
import numpy as np

# Convert lists to NumPy arrays first
X_array = np.array(X, dtype=np.float32)
hour_array = np.array(hour_list, dtype=np.int64)
day_array = np.array(day_list, dtype=np.int64)
month_array = np.array(month_list, dtype=np.int64) - 1  # Ensure months range from 0 to 11
holiday_array = np.array(holiday_list, dtype=np.int64)
workday_array = np.array(workday_list, dtype=np.int64)
y_array = np.array(y, dtype=np.float32)

# Convert NumPy arrays to PyTorch tensors
X_tensor = torch.from_numpy(X_array)
hour_tensor = torch.from_numpy(hour_array)
day_tensor = torch.from_numpy(day_array)
month_tensor = torch.from_numpy(month_array)
holiday_tensor = torch.from_numpy(holiday_array)
workday_tensor = torch.from_numpy(workday_array)
y_tensor = torch.from_numpy(y_array)

In [39]:
import sys
import os

# Add the parent directory to Python's search path
sys.path.append(os.path.abspath("../models/tranformers"))

# Now you can import Autoformer
from autoformer import Autoformer


# Example usage
seq_length = 20  # Number of time steps
batch_size = 32
input_dim = 6
hidden_dim = 64
output_dim = 1

# Initialize model and test prediction
model = Autoformer(input_dim, hidden_dim, output_dim)
output = model(X_tensor, hour_tensor, day_tensor, month_tensor, holiday_tensor, workday_tensor)
print("Output shape:", output.shape)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4025600x33 and 6x64)